# Nama: Vinsensius Carlos
# NIM: 4222301072
# Kelas: Robotika C Pagi

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

## Load Our Dataset

In [ ]:
df = pd.read_csv('WineQT.csv')

## EDA

In [ ]:
fig, ax = plt.subplots(figsize=(15,10))
sns.scatterplot(data=df, x='alcohol', y='quality')
plt.title('Sebaran Nilai alcohol vs quality')

In [ ]:
df.describe()

In [ ]:
columns = ['alcohol', 'volatile acidity', 'citric acid', 'sulphates']

for col in columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=col, data=df)
    plt.title(f'Box Plot - {col}')
    plt.xlabel(col)
    plt.ylabel('Nilai')
    plt.show()

In [ ]:
for col in columns:
    plt.figure(figsize=(12, 6))
    sns.histplot(df[col], bins=30, kde=False)
    plt.title(f'Histogram - {col}')
    plt.xlabel(col)
    plt.ylabel('Frekuensi')
    plt.show()

## Feature Engineering
* Unsupervised tidak perlu dilakukan splitting
1. Drop Duplikat
2. Outlier Handling (opsional)
3. Feature Scalling

In [ ]:
print(f"Dataframe dimension before duplication drop {df.shape[0]}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dataframe dimension after duplication drop {df.shape[0]}")

In [ ]:
fitur_columns = ['alcohol', 'volatile acidity', 'citric acid', 'sulphates',
                 'fixed acidity', 'residual sugar', 'chlorides',
                 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH']
X = df[fitur_columns].values
y = df['quality'].values  # label kualitas wine

In [ ]:
from sklearn.preprocessing import StandardScaler
X_std = StandardScaler().fit_transform(X)
df_scalling = pd.DataFrame(data=X_std, columns=fitur_columns)
df_scalling.describe()

In [ ]:
X_std

In [ ]:
df_scalling

## TO DO!
- Lengkapi Code dibawah ini, untuk mengecek distribusi sebelum dan setelah dilakukan feature scalling menggunakan standar scaller

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# Plot distribusi sebelum Standar Scaler
sns.histplot(df['alcohol'], bins=30, kde=True, ax=ax1, color='blue', label='Alcohol Asli')
sns.histplot(df['volatile acidity'], bins=30, kde=True, ax=ax1, color='red', label='Volatile Acidity Asli')
ax1.set_title('Distribusi Sebelum Standard Scaler')
ax1.legend()

# Plot distribusi setelah Standar Scaler
sns.histplot(df_scalling['alcohol'], bins=30, kde=True, ax=ax2, color='blue', label='Alcohol Scaled')
sns.histplot(df_scalling['volatile acidity'], bins=30, kde=True, ax=ax2, color='red', label='Volatile Acidity Scaled')
ax2.set_title('Distribusi Setelah Standard Scaler')
ax2.legend()

plt.tight_layout()
plt.show()

Terlihat perubahan skala sebelum menggunakan standar scaller dan sesudah standard scaller walau masih memiliki pola yang sama. Terlihat juga distribusinya lebih simetris seperti distribusi normal.

## K-means Clustering
Pada pembahasan kali ini akan diuji 2 metode pemilihan nilai cluster (K) yang terbaik, mendekati distribusi pada label kualitas wine.
1. Metode Elbow
2. Via-Score Plot

### Metode Elbow

In [ ]:
from sklearn.cluster import KMeans
inertia = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=0)
    kmeans.fit(df_scalling.values)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(20, 10))
sns.lineplot(x=range(1, 11), y=inertia, color='#000087', linewidth=4)
sns.scatterplot(x=range(1, 11), y=inertia, s=300, color='#800000', linestyle='--')
plt.title('Elbow Method - Finding K value Clusters')
plt.xlabel('The Amount of K')
plt.ylabel('Intertia / WSS')

In [ ]:
# Dari hasil diatas elbow dipilih pada angka 3
from sklearn.cluster import KMeans
kmeans_elbow = KMeans(n_clusters=3, random_state=0)
kmeans_elbow.fit(df_scalling.values)

In [ ]:
# Taruh hasil k-means elbow method ke df dengan nama kolom cluster_elbow
df['cluster_elbow'] = kmeans_elbow.labels_

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(figsize=(15,10))
sns.scatterplot(data=df, x='alcohol', y='quality', hue='cluster_elbow')
plt.title('K-Means Elbow - alcohol vs quality')

### Bandingkan hasil dengan label kualitas wine

In [ ]:
fig, ax = plt.subplots(figsize=(15,10))
sns.scatterplot(data=df, x='alcohol', y='quality', hue='quality', palette='tab10')
plt.title('Distribusi Label Kualitas Wine Asli')

### Hasil diatas ketika menggunakan elbow ialah tidak optimal, karena cluster tidak selalu cocok dengan label kualitas asli. Dengan kondisi :
1. Cluster 0 -> Wine kualitas rendah
2. Cluster 1 -> Wine kualitas sedang
3. Cluster 2 -> Wine kualitas tinggi

### 2. Via Score Plot

In [ ]:
!pip install yellowbrick

In [ ]:
from yellowbrick.cluster import KElbowVisualizer
k_means_via = KMeans()
visualizer = KElbowVisualizer(k_means_via, k=(1,11), timings=True)
visualizer.fit(df_scalling.values)
plt.title('Distortion Via-Score Plot for K-Means Clustering')
plt.show()

In [ ]:
# Dari hasil diatas K dipilih pada angka 3
from sklearn.cluster import KMeans
kmeans_via = KMeans(n_clusters=3, random_state=0)
kmeans_via.fit(df_scalling.values)

In [ ]:
# Taruh hasil k-means via score method ke df dengan nama kolom cluster_via
df['cluster_via'] = kmeans_via.labels_

In [ ]:
df

## TO DO !
- Lakukan evaluasi pada k-means menggunakan via score secara visualisasi

In [ ]:
# Lakukan evaluasi setelah dilakukan K-Means
# Bandingkan hasil cluster dengan distribusi data asli
fig, ax = plt.subplots(figsize=(15,10))
sns.scatterplot(data=df, x='alcohol', y='quality', hue='cluster_via')
plt.title('K-Means Via-Score - alcohol vs quality')

### Bandingkan dengan label kualitas wine

In [ ]:
fig, ax = plt.subplots(figsize=(15,10))
sns.scatterplot(data=df, x='alcohol', y='quality', hue='quality', palette='tab10')
plt.title('Label Kualitas Wine Asli')

### Tulis Interpretasi dari hasil evaluasi diatas :
### Isi disini
1. Meskipun k-means berhasil membagi data wine menjadi 3 cluster, cluster yang terbentuk tidak selalu selaras dengan label kualitas wine asli (skala 3-8).
2. Cluster dengan kadar alcohol tinggi cenderung mengelompok bersama wine berkualitas lebih tinggi.
3. Volatile acidity yang tinggi cenderung berkaitan dengan wine berkualitas rendah, sehingga fitur ini cukup informatif dalam pembentukan cluster.

Perbedaan antara hasil clustering dan label asli terjadi karena k-means hanya mempertimbangkan kedekatan jarak antar fitur numerik, sementara label kualitas wine merupakan penilaian subjektif yang melibatkan kombinasi kompleks dari banyak fitur kimia.

### Thank you :)